# nanoDiffusionCodex Colab Training

This notebook prepares a small Hugging Face code dataset, trains the masked-token diffusion model, evaluates masked-token perplexity, and samples a short completion. Use a GPU runtime for the training cells.

In [ ]:
# Optional: mount Drive if you want checkpoints to persist after the runtime stops.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# If this notebook is not already running from the repo root, set REPO_URL and clone it.
import os
from pathlib import Path

REPO_URL = ""  # Example: "https://github.com/<owner>/nanoDiffusionCodex.git"
repo_root = Path.cwd()
if not (repo_root / "scripts" / "train.py").exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL or upload the repository, then run from the repo root.")
    !git clone $REPO_URL /content/nanoDiffusionCodex
    os.chdir('/content/nanoDiffusionCodex')
else:
    os.chdir(repo_root)

print('repo:', Path.cwd())

In [ ]:
!python -m pip install -q -r requirements.txt
!python - <<'PY'
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
PY

In [ ]:
# Choose one: "codeparrot-clean" or "codesearchnet-python".
DATASET_PRESET = "codeparrot-clean"
MAX_SAMPLES = 2048
VAL_SAMPLES = 256
MAX_SEQ_LEN = 256

!python scripts/prepare_hf_dataset.py \
  --preset $DATASET_PRESET \
  --max-samples $MAX_SAMPLES \
  --val-samples $VAL_SAMPLES \
  --max-seq-len $MAX_SEQ_LEN \
  --output-dir data/processed

In [ ]:
# Optionally patch a shorter Colab run without editing the checked-in default config.
import copy, yaml
from pathlib import Path

cfg = yaml.safe_load(Path('src/nano_diffusion/configs/default.yaml').read_text())
cfg['model']['max_seq_len'] = MAX_SEQ_LEN
cfg['model']['dim'] = 128
cfg['model']['layers'] = 4
cfg['model']['heads'] = 4
cfg['training']['batch_size'] = 16
cfg['training']['total_steps'] = 500
cfg['training']['eval_interval'] = 100
cfg['training']['save_interval'] = 250
cfg['training']['output_dir'] = 'runs/colab-byte-diffusion'
Path('configs_colab.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path('configs_colab.yaml').read_text())

In [ ]:
!python scripts/train.py --config configs_colab.yaml

In [ ]:
!python scripts/eval.py --checkpoint runs/colab-byte-diffusion/best.pt --batch-size 16

In [ ]:
!python scripts/infer.py \
  --checkpoint runs/colab-byte-diffusion/best.pt \
  --prompt "def add(a, b):\n    " \
  --new-tokens 96 \
  --steps 16

In [ ]:
# Optional: persist checkpoints to Drive.
if MOUNT_DRIVE:
    !mkdir -p /content/drive/MyDrive/nanoDiffusionCodex
    !cp -r runs/colab-byte-diffusion /content/drive/MyDrive/nanoDiffusionCodex/